# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    base_delay = 0.5
    last = None
    for attempt in range(tries):
        try:
            r = requests.get(BASE, params=params, timeout=15)
            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])
            if r.status_code not in RETRY_STATUSES:
                raise RuntimeError(
                    f'not retryable: {r.status_code} {r.text[:200]}')
            last = f'status {r.status_code}'
        except requests.RequestException as e:
            last = repr(e)
        delay = base_delay * (2 ** attempt)
        print(f'attempt {attempt + 1} failed ({last}); retrying in {delay}s')
        time.sleep(delay)
    raise RuntimeError(f'gave up after {tries} attempts: {last}')

This code takes a start and end date, a latititude and a longitude, and a  `tries` count that establishes that it will make 3 attempts. Then, it builds a `params` dictionary. This holds information that the API will need to know, the location, dates, daily variables, and timezone that we want to look at.

Before the loop, `base_delay` is set to 0.5 seconds, and `last` is created as an empty placeholder that will be able to give a final error message that will tell us what went wrong.

Then, the for loop runs once per attempt. The `try` block calls `requests.get` with `BASE`, the parameters, and a 15-second time out so the code doesn't spend excessive time waiting on a server. If the status code is 200, the request was succesful, and the function will create a data frame out of the daily variables.

If the status is not a 200, the code checks if it's in `RETRY_STATUSES`, which holds 429 and 5xx server errors. If the error is a 400 or a 404, the problem is the request rather than the server, so there is no reason to retry. A 400 or a 404 will raise a `RuntimeError` with the status and the first 200 characters of the response, and everything will stop at once. If the status is in the retryable set, the code records `last = 'status XXX'` with whichever error code, and continues.


The `except requests.RequestException` looks at network-level problems such as a dropped connection. It stores a description of the error that happened in `last` and continues the loop, since those problems are usually temporary.

When an attempt fails in a retryable way, the code goes to the `delay` lines. The delay is `base_delay * 2 ** attempt`, which gives .5 seconds after the first failure, 1 second after the second, and 2 after the third. This allows a server time to recover. Then, the code prints a message showing which attempt failed and how long it will wait.

Finally, if the loop finishes without returning, all attempts failed and a `RuntimeError` will be raised saying who many attempts were made, and looks at `last` to see the explanation for the final failure.


### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
from datetime import datetime
from zoneinfo import ZoneInfo

used_fallback = False
pulled_at = None

DATES = ['2026-09-05', '2026-10-03']

try:
    weather = get_weather(DATES[0], DATES[1])
    pulled_at = datetime.now(ZoneInfo('America/New_York'))   # when the pull succeeded
    weekends = weather[weather['time'].isin(DATES)].reset_index(drop=True)
    print(weekends)
except Exception as e:
    print('get_weather failed:', e)
    used_fallback = True
    weekends = None

         time  temperature_2m_max  precipitation_sum
0  2026-09-05                29.4                2.5
1  2026-10-03                22.0               14.7


The code starts with `used_fallback = False`, which records if you have to use the fallback in part 3, rather than live data. It begins as False because we haven't needed it yet, and the except block is where that could change.

`pulled_at = None` is set up in the same way. It will hold the time the data was pulled for a question later in the notebook, and it stays `None` unless the request succeeds.

`DATES` is a ist holding the dates we want to look at, in the string format API uses for its time column.

Then, the `try` block calls `get_weather` once, with the first date as the start and the second as the end. This makes one request with data from September 5th through October 3rd, which allows us to only make one request rather than having to make two separate requests for each date. This results in a data frame with one row per day.

Right after `get_weather` returns, `pulled_at = datetime.now(ZoneInfo('America/New_York'))` saves the current time in Eastern time. It's the line immediately after the call so it can record when the data was actually fetched. Because the call raises an exception if it fails, this line only runs when the pull succeeded.

Because we only want two of those rows, the next line filters for only the two dates we want. `weekends` is created and `reset_index(drop=True)` renumbers the rows. This data frame is printed so we can see the results.

Then, if `get_weather` fails, it raises an exception. It will retry on its own, and will raise an error when it has given up or hit a non-retryable status. By the time the exception hits the except, the call has failed. Then, the error message is printed, and sets `used_fallback = True` to show that we are using the fall

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

if used_fallback:
    wx = FALLBACK.copy()
    wx['source'] = 'fallback sample'
else:
    wx = weekends.copy()
    wx['source'] = 'open-meteo'

print(wx)

         time  temperature_2m_max  precipitation_sum      source
0  2026-09-05                29.4                2.5  open-meteo
1  2026-10-03                22.0               14.7  open-meteo


This code defines `FALLBACK`, which has sample values for the 2 dates. This is so we can run the notebook even when there's no network, but these numbers are made up so we need to label them.

Then, we make an `if` statement that checks `used_fallback`. This flag was established in part 2, so it will only flip to True if `get_weather` raised an exception if the live call failed.

If it does fail, `FALLBACK.copy()` makes a separate copy of the sample table and stores it in `wx`. Then, `wx['source']= 'fallback sample'` adds a new column and puts that label in every row. We have to make a separate copy because we don't want to directly change `FALLBACK` itself.  

If the call succeeded, the `else` branch runs and copies `weekends` with the real data, and labels every row `open-meteo`. Copying is needed because adding a column to a filtered table may trigger a pandas warning.

The end result is the `wx` table with a column telling us where the data came from. `wx` is printed to see results.

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})
wx['date'] = pd.to_datetime(wx['date'])
wx = wx[['date', 'temp_max', 'precip_mm', 'source']]

print(wx)
print(wx.dtypes)

        date  temp_max  precip_mm      source
0 2026-09-05      29.4        2.5  open-meteo
1 2026-10-03      22.0       14.7  open-meteo
date         datetime64[ns]
temp_max            float64
precip_mm           float64
source               object
dtype: object


This code reshapes the `wx` table from Part 3 so it matches our own naming, and saves the result back into `wx`.

We first have to rename our columns. We use a dictionary where each key is the old column name, and each value is the new one. `time` becomes `date`, `temperature_2m_max` becomes `temp_max`, and `precipitation_sum` becomes `precip_mm`. The source column isn't in the dictionary because it already has the right name.

Then, `pd.to_datetime` converts the date column to the correct data type. Before, it was a string. Converting it allows them to be datetime values, which can be sorted, compared, and joined to other datetime columns.

Then, `wx[['date', 'temp_max', 'precip_mm', 'source']]` selects those four columns in that order to create the final data frame.

### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [5]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [6]:
wetter = wx.loc[wx['precip_mm'].idxmax()]
drier = wx.loc[wx['precip_mm'].idxmin()]
diff = wetter['precip_mm'] - drier['precip_mm']

print(
    f"{wetter['date']:%B %d, %Y} was wetter than {drier['date']:%B %d, %Y}, "
    f"with {wetter['precip_mm']:.1f} mm of rain versus {drier['precip_mm']:.1f} mm, "
    f"a difference of {diff:.1f} mm."
)

October 03, 2026 was wetter than September 05, 2026, with 14.7 mm of rain versus 2.5 mm, a difference of 12.2 mm.


The code starts by finding the two rows we care about. `wx['precip_mm'].idxmax()` finds the row with the highest precipitation value, and `.idxmin()` finds the row with the lowest. Now, `wetter` and `drier` hold the entirety of each row for both of those days.

Then, `diff` subtracts the smaller rainfall from the larger one, and saves this number for the print statement.

The print statement builds a sentence out of all of the pieces we found with the previous code.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [7]:
LAT, LON = 38.03, -78.51   # same values as the defaults in get_weather

if used_fallback:
    pulled = 'no successful pull (live request failed)'
    data_status = 'FALLBACK sample values, not real measurements'
else:
    pulled = f"{pulled_at:%Y-%m-%d %H:%M %Z}"
    data_status = 'real data from the Open-Meteo archive'

print(f"Endpoint:    {BASE}")
print(f"Coordinates: {LAT}, {LON}")
print(f"Date range:  {DATES[0]} to {DATES[1]} (only these two days kept)")
print(f"Pulled:      {pulled}")
print(f"Data:        {data_status}")

Endpoint:    https://archive-api.open-meteo.com/v1/archive
Coordinates: 38.03, -78.51
Date range:  2026-09-05 to 2026-10-03 (only these two days kept)
Pulled:      2026-10-11 17:03 EDT
Data:        real data from the Open-Meteo archive


This cell starts by defining `LAT` and `LON` to be the same coordinates `get_weather` used. We have to define them because they were previously only inside the function, so the notebook has no variable to print.

Then, there's an `if` that looks at `used_fallback`. It sets two variables, `pulled` and `data_status`, which are two lines of the print line that will depend on if the call worked or not. If the fallback was used, no data was fetched, so `pulled` will say there was no succesful pull and `data_status` will tell us the numbers were sample values.

If the call succeeded, the `else` branch runs. `pulled` is built from `pulled_at` from part 2, which saved the time right after `get_weather` returned. `data_status` tells us that they data came from the Open-Meteo archive.

Then the five print lines tell us all this information in a readable format.

### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If `get_weather` returned an empty DataFrame on failure instead of raising, the `try` block doesn't reach the `except`. This would result in `used_fallback` saying False, and `pulled_at` would record a timestamp for a pull that didn't return any actual information. Part 3 would label the result `'open-meteo'`, and we would think real data was pulled.

The merge would be silent, because it's valid to merge on an empty table so no error would be raised. If it was a left merge, all 200 rows would be merged, but `temp_max` and `precip_mm` would be missing in each row. If it was an inner merge, there would be zero rows. Neither merge would produce an error, and it if no one looks at the row count for an inner merge or what the rows look like for a left merge, no one would catch it because there's no error code.


A chart with rain against revenue would be problematic because the plotting libraries drop `NaN` points automatically, so the left merge would result in a scatter with no dots. It would be even worse if someone used `.fillna(0)` to fill in missing values. This would show all 200 days have no rain, and the chart would be a vertical stack of points with varying revenue.


No one would catch this error because there is no error raised, so every step technically succeeds. The wrong answer could be plausible (no rainy days, rain doesn't impact revenue), so people may not be immediately alerted. And because the source column and citation cell say the data is real, readers would believe this data is correct.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [8]:
ORLANDO = (28.54, -81.38)

raw = get_weather(DATES[0], DATES[1], lat=ORLANDO[0], lon=ORLANDO[1])
orl = raw[raw['time'].isin(DATES)].rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})
orl['date'] = pd.to_datetime(orl['date'])
orl['source'] = 'open-meteo'

cville = wx.copy()

cville['city'] = 'Charlottesville'
orl['city'] = 'Orlando'

weather_long = pd.concat([cville, orl], ignore_index=True)
weather_long = weather_long[['city', 'date', 'temp_max', 'precip_mm', 'source']]

print(weather_long)

              city       date  temp_max  precip_mm      source
0  Charlottesville 2026-09-05      29.4        2.5  open-meteo
1  Charlottesville 2026-10-03      22.0       14.7  open-meteo
2          Orlando 2026-09-05      32.1        3.9  open-meteo
3          Orlando 2026-10-03      30.4       14.5  open-meteo


First, `ORLANDO` is defined with a pair of the latitutde and longitude  of Orlando.

Then, Orland's weather is pulled using `get_weather`. `get_weather` is called with the same two dates as before, with the new lat and lon values for Orlando. It works the same as part 2, so the one request covers the whole range of dates. From this, we get `raw`, a table with rows for 29 days.

`raw['time'].isin(DATES)` keeps only the two dates that we want to look at. Then, the columns are renamed to follow the naming convention previously established to match the previously made data frame. `pd.to_datetime` turns the date column into datetime values. `orl['source'] = 'open-meteo'` labels the rows.

Then, `cville=wx.copy()` makes a separate copy so adding a column won't change the original `wx` data frame.

Each frame gets a `city` column, either Charlottesville or Orlando, so each row is labeled. This allows for the table to be in long format, so there is one row per city per date and the city columnn tells us which is which.

`pd.concat([cville, orl], ignore_index=True)` stacks our two data frames on top of one another. `ignore_index=True` renumbers the rows to be 0-3. Then, the columns are selected in a fixed order.

Then the final `weather_long` data frame is printed.